# 🚔 PatrolIQ — Data Preprocessing Pipeline

### What this notebook does
This notebook walks through the **complete data cleaning and feature engineering pipeline** used in the PatrolIQ Smart Safety Analytics Platform.

We start from the **raw Chicago Crime dataset (500K+ records)** and produce a clean, analysis-ready CSV with engineered features that feed our ML models.

---
### 📌 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Load Raw Data](#2-load)
3. [Data Exploration (Before Cleaning)](#3-explore)
4. [Data Cleaning](#4-clean)
5. [Feature Engineering](#5-features)
6. [Data Quality Checks](#6-quality)
7. [Save Cleaned Data & Metadata](#7-save)

---
> **Dataset:** [City of Chicago — Crimes 2001 to Present](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2)  
> **Records:** ~507,937 rows | **Columns:** 22 original → 30 after engineering

## 1. Setup & Imports <a id='1-setup'></a>

We import only standard scientific Python libraries:
- `pandas` — tabular data manipulation
- `numpy` — numerical operations
- `os / json` — file system and metadata storage
- `warnings` — suppress non-critical sklearn/pandas warnings

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

# ── File Paths ─────────────────────────────────────────────────────────────────
# Go one level up from notebooks/ to reach the project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_PATH     = os.path.join(PROJECT_ROOT, 'data', 'uncleaned', 'crimes_data.csv')
CLEAN_DIR    = os.path.join(PROJECT_ROOT, 'data', 'cleaned')
os.makedirs(CLEAN_DIR, exist_ok=True)

print('✅ Imports successful')
print(f'📂 Raw data   : {RAW_PATH}')
print(f'📂 Output dir : {CLEAN_DIR}')

## 2. Load Raw Data <a id='2-load'></a>

The raw CSV contains **507,937 rows** and **22 columns** including:
- **ID / Case Number** — unique crime identifiers
- **Date** — timestamp of crime occurrence
- **Primary Type / Description** — crime category (e.g., THEFT, BATTERY)
- **Location Description** — where it happened (e.g., STREET, APARTMENT)
- **Arrest / Domestic** — boolean flags
- **Latitude / Longitude** — geographic coordinates
- **District / Beat / Ward / Community Area** — Chicago administrative divisions

> `low_memory=False` prevents pandas from making incorrect dtype guesses on large files.

In [ ]:
print('Loading raw CSV — this may take 15-30 seconds for 500K rows...')
df_raw = pd.read_csv(RAW_PATH, low_memory=False)

print(f'\n✅ Loaded successfully!')
print(f'   Shape     : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'   Memory    : {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df_raw.head(3)

## 3. Data Exploration — Before Cleaning <a id='3-explore'></a>

Before cleaning, we understand the data's structure — missing values, data types, and unique categories.

In [ ]:
# ── Column Data Types ──────────────────────────────────────────────────────────
print('=== Column Info ===')
df_raw.info()

In [ ]:
# ── Missing Values Analysis ────────────────────────────────────────────────────
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print('=== Missing Values Report ===')
print(missing_report[missing_report['Missing Count'] > 0])

In [ ]:
# ── Crime Type Distribution (Top 15) ──────────────────────────────────────────
# Understanding what types of crimes dominate the dataset
print('=== Top 15 Crime Types ===')
crime_counts = df_raw['Primary Type'].value_counts().head(15)
print(crime_counts.to_string())

print(f'\nTotal unique crime types: {df_raw["Primary Type"].nunique()}')

In [ ]:
# ── Date Range ────────────────────────────────────────────────────────────────
# How far back does the data go?
# Note: We parse the date here just for exploration; full parsing happens in cleaning
sample_dates = pd.to_datetime(df_raw['Date'].dropna().sample(1000, random_state=42),
                              format='%m/%d/%Y %I:%M:%S %p', errors='coerce')

print(f'Date range (sample): {sample_dates.min()} → {sample_dates.max()}')
print(f'Arrest rate (raw): {df_raw["Arrest"].value_counts(normalize=True).to_dict()}')

## 4. Data Cleaning <a id='4-clean'></a>

We apply three cleaning steps:

| Step | What it does | Why |
|------|--------------|-----|
| **Drop nulls** | Remove rows missing Latitude, Longitude, Primary Type, or Date | ML models can't handle NaN inputs |
| **Geo filter** | Keep only points within Chicago's bounding box (41.5°–42.1°N, 87.4°–87.95°W) | Remove erroneous (0,0) or out-of-city entries |
| **Parse datetime** | Convert 'Date' string to pandas datetime | Required for time-based feature extraction |

In [ ]:
# Work on a copy so we can always restart from df_raw
df = df_raw.copy()

# ── Step 1: Drop rows with missing critical values ─────────────────────────────
# These four columns are ESSENTIAL — without them a record is useless for analysis
critical_cols = ['Latitude', 'Longitude', 'Primary Type', 'Date']
before = len(df)
df = df.dropna(subset=critical_cols)
print(f'Step 1 — Drop nulls   : {before:,} → {len(df):,} rows  ({before-len(df):,} removed)')

# ── Step 2: Geographic Bounding Box Filter ─────────────────────────────────────
# Chicago roughly spans: Lat 41.5–42.1, Lon -87.95 to -87.4
# Anything outside is a data entry error
before = len(df)
df = df[
    df['Latitude'].between(41.5, 42.1) &
    df['Longitude'].between(-87.95, -87.4)
]
print(f'Step 2 — Geo filter   : {before:,} → {len(df):,} rows  ({before-len(df):,} removed)')

# ── Step 3: Parse DateTime ─────────────────────────────────────────────────────
# Original format: '01/01/2024 01:30:00 AM'
# errors='coerce' turns any unparseable string into NaT (Not a Time)
before = len(df)
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df = df.dropna(subset=['Date'])   # Remove rows where datetime parsing failed
print(f'Step 3 — Parse dates  : {before:,} → {len(df):,} rows  ({before-len(df):,} removed)')

print(f'\n✅ Cleaning complete! Shape: {df.shape}')

## 5. Feature Engineering <a id='5-features'></a>

**Feature engineering** creates new meaningful columns from existing raw data.  
These features help the ML models discover patterns that aren't visible in the raw columns.

### Features we create:

| Feature | Source | Purpose |
|---------|--------|---------|
| `Hour` | `Date.hour` | Crime peaks by hour (rush hour vs midnight?) |
| `Day_of_Week` | `Date.day_name()` | Weekend vs weekday patterns |
| `Day_Num` | Map Monday=0…Sunday=6 | Numeric encoding for ML |
| `Month` | `Date.month` | Seasonal crime variation |
| `Year` | `Date.year` | Year-over-year trends |
| `Season` | Map month → season | Winter/Spring/Summer/Fall |
| `Is_Weekend` | Sat or Sun? | Binary weekend flag |
| `Is_Night` | Hour ≥ 20 or < 6 | Night crime indicator |
| `Crime_Severity_Score` | Domain knowledge map | 1–10 score per crime type |

### Crime Severity Score (Domain Knowledge)
We assigned a severity score (1–10) to each crime type based on **public safety impact**:
- **10** = Homicide (most severe)
- **1** = Non-criminal / minor infractions
This converts a categorical variable into a meaningful numeric feature.

In [ ]:
# ── Severity Score Mapping ────────────────────────────────────────────────────
# Domain-knowledge-based severity on a 1-10 scale
# Higher score = more severe / dangerous crime
SEVERITY_MAP = {
    # Tier 5 — Most Severe (9-10)
    'HOMICIDE': 10,
    'KIDNAPPING': 9,
    'HUMAN TRAFFICKING': 9,

    # Tier 4 — Severe (7-8)
    'CRIMINAL SEXUAL ASSAULT': 8,
    'CRIM SEXUAL ASSAULT': 8,       # alternate spelling in the dataset
    'SEX OFFENSE': 7,
    'ROBBERY': 7,
    'ARSON': 7,

    # Tier 3 — Moderate-High (5-6)
    'ASSAULT': 6,
    'BATTERY': 6,
    'STALKING': 6,
    'INTIMIDATION': 5,
    'BURGLARY': 5,
    'MOTOR VEHICLE THEFT': 5,
    'WEAPONS VIOLATION': 5,
    'OFFENSE INVOLVING CHILDREN': 5,

    # Tier 2 — Moderate (3-4)
    'NARCOTICS': 4,
    'OTHER NARCOTIC VIOLATION': 4,
    'PROSTITUTION': 4,
    'CRIMINAL DAMAGE': 3,
    'CRIMINAL TRESPASS': 3,
    'DECEPTIVE PRACTICE': 3,
    'THEFT': 3,
    'FRAUD': 3,
    'FORGERY & COUNTERFEITING': 3,

    # Tier 1 — Minor (1-2)
    'INTERFERENCE WITH PUBLIC OFFICER': 2,
    'LIQUOR LAW VIOLATION': 2,
    'GAMBLING': 2,
    'PUBLIC PEACE VIOLATION': 2,
    'OBSCENITY': 2,
    'OTHER OFFENSE': 2,
    'CONCEALED CARRY LICENSE VIOLATION': 2,
    'NON-CRIMINAL': 1,
    'NON - CRIMINAL': 1,
}

print(f'Severity mapping covers {len(SEVERITY_MAP)} crime types')

In [ ]:
# ── Day-of-Week Name → Number ─────────────────────────────────────────────────
# ML models work with numbers, not strings like 'Monday'
DAYNUM_MAP = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
    'Friday': 4, 'Saturday': 5, 'Sunday': 6
}

# ── Season Mapping ─────────────────────────────────────────────────────────────
# Chicago has distinct crime patterns across seasons
SEASON_MAP = {
    12: 'Winter', 1: 'Winter',  2: 'Winter',
     3: 'Spring', 4: 'Spring',  5: 'Spring',
     6: 'Summer', 7: 'Summer',  8: 'Summer',
     9: 'Fall',  10: 'Fall',   11: 'Fall'
}

# ── Temporal Features ─────────────────────────────────────────────────────────
# Extract granular time components from the datetime column
df['Hour']        = df['Date'].dt.hour          # 0-23
df['Day_of_Week'] = df['Date'].dt.day_name()    # 'Monday', 'Tuesday', ...
df['Day_Num']     = df['Day_of_Week'].map(DAYNUM_MAP).fillna(0).astype(int)  # 0-6
df['Month']       = df['Date'].dt.month         # 1-12
df['Year']        = df['Date'].dt.year          # 2003-2026
df['Season']      = df['Month'].map(SEASON_MAP) # Winter/Spring/Summer/Fall

# ── Binary Flags ──────────────────────────────────────────────────────────────
# astype(int) converts True→1, False→0 for ML compatibility
df['Is_Weekend']  = df['Day_of_Week'].isin(['Saturday', 'Sunday']).astype(int)
df['Is_Night']    = ((df['Hour'] >= 20) | (df['Hour'] < 6)).astype(int)

# ── Severity Score ────────────────────────────────────────────────────────────
# .map() looks up each crime type in our dictionary
# .fillna(2) assigns score=2 to any unmapped crime types (default: minor)
df['Crime_Severity_Score'] = df['Primary Type'].map(SEVERITY_MAP).fillna(2).astype(int)

print('Temporal & severity features created!')
df[['Date', 'Hour', 'Day_of_Week', 'Month', 'Year', 'Season',
    'Is_Weekend', 'Is_Night', 'Crime_Severity_Score']].head(5)

In [ ]:
# ── Encode Boolean Columns as Integers ───────────────────────────────────────
# The raw CSV stores Arrest/Domestic as 'true'/'false' strings
# We convert them to 1/0 integers for ML compatibility
for col in ['Arrest', 'Domestic']:
    if df[col].dtype == object:
        # String: 'true' → 1, 'false' → 0
        df[col] = df[col].str.lower().map({'true': 1, 'false': 0}).fillna(0).astype(int)
    else:
        # Already boolean/numeric, just ensure int type
        df[col] = df[col].astype(int)

# ── Encode Administrative Areas as Integers ───────────────────────────────────
# District, Beat, Ward, Community Area are numeric IDs
# pd.to_numeric(errors='coerce') safely handles any stray text → NaN → 0
for col in ['District', 'Beat', 'Ward', 'Community Area']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print('Encoding complete!')
print(f'Arrest   — unique values: {df["Arrest"].unique()}')
print(f'Domestic — unique values: {df["Domestic"].unique()}')
print(f'District — range: {df["District"].min()} to {df["District"].max()}')
print(f'\nFinal dataset shape: {df.shape}')

## 6. Data Quality Checks <a id='6-quality'></a>

After all transformations, we verify the data is clean and ready for ML.

In [ ]:
print('=== FINAL DATASET SUMMARY ===')
print(f'Total rows    : {len(df):,}')
print(f'Total columns : {len(df.columns)}')
print(f'Memory usage  : {df.memory_usage(deep=True).sum()/1e6:.1f} MB')

print('\n=== Remaining Missing Values ===')
remaining_nulls = df.isnull().sum()
if remaining_nulls.sum() == 0:
    print('✅ No missing values in critical numeric columns!')
else:
    print(remaining_nulls[remaining_nulls > 0])

In [ ]:
# ── Key Statistics ─────────────────────────────────────────────────────────────
print('=== Key Dataset Statistics ===')
print(f'Year range        : {df["Year"].min()} — {df["Year"].max()}')
print(f'Crime types       : {df["Primary Type"].nunique()} unique categories')
print(f'Districts         : {df["District"].nunique()} police districts')
print(f'Arrest rate       : {df["Arrest"].mean()*100:.2f}%')
print(f'Domestic rate     : {df["Domestic"].mean()*100:.2f}%')
print(f'Night crimes      : {df["Is_Night"].mean()*100:.2f}%')
print(f'Weekend crimes    : {df["Is_Weekend"].mean()*100:.2f}%')
print(f'Avg severity score: {df["Crime_Severity_Score"].mean():.2f} / 10')

In [ ]:
# ── Severity Score Distribution ───────────────────────────────────────────────
sev_dist = df.groupby('Crime_Severity_Score')['Primary Type'].agg(
    Count='count',
    Example=lambda x: x.value_counts().index[0]
).reset_index()

print('=== Crime Severity Distribution ===')
print(sev_dist.to_string(index=False))

## 7. Save Cleaned Data & Metadata <a id='7-save'></a>

We save:
1. **`cleaned_crimes.csv`** — the full cleaned dataset used by all ML steps
2. **`metadata.json`** — dataset summary for the Streamlit app dashboard (KPI cards)

In [ ]:
# ── Save Metadata JSON ────────────────────────────────────────────────────────
# This JSON powers the KPI cards on the Streamlit Home page
meta = {
    'total_records' : int(len(df)),
    'crime_types'   : sorted(df['Primary Type'].unique().tolist()),
    'years'         : sorted(df['Year'].unique().tolist()),
    'districts'     : sorted(df['District'].unique().tolist()),
    'arrest_rate'   : float(round(df['Arrest'].mean() * 100, 2)),
    'domestic_rate' : float(round(df['Domestic'].mean() * 100, 2)),
    'columns'       : df.columns.tolist(),
}

meta_path = os.path.join(CLEAN_DIR, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'✅ metadata.json saved → {meta_path}')
print(f'   Crime types: {len(meta["crime_types"])}')
print(f'   Years      : {meta["years"][0]} — {meta["years"][-1]}')
print(f'   Arrest rate: {meta["arrest_rate"]}%')

In [ ]:
# ── Save Cleaned CSV ──────────────────────────────────────────────────────────
# index=False: don't write the pandas row index as a column
out_path = os.path.join(CLEAN_DIR, 'cleaned_crimes.csv')
df.to_csv(out_path, index=False)

print(f'✅ cleaned_crimes.csv saved → {out_path}')
print(f'   Shape   : {df.shape}')
print(f'   Columns : {df.columns.tolist()}')

# Show a 5-row sample to confirm structure
df.sample(5, random_state=42)[[
    'Primary Type', 'Date', 'Hour', 'Day_of_Week', 'Season',
    'Is_Night', 'Is_Weekend', 'Crime_Severity_Score',
    'Latitude', 'Longitude', 'Arrest', 'District'
]]

---
## ✅ Preprocessing Complete!

### Summary of what we did:

| Step | Input | Output |
|------|-------|--------|
| Load raw data | 507,937 rows, 22 columns | 507,937 rows |
| Drop nulls | — | ~500K rows (removed missing geo/date) |
| Geo filter | — | ~505K rows (removed out-of-Chicago) |
| Feature engineering | 22 columns | 30 columns (+8 new features) |
| **Final output** | — | **~505,171 rows × 30 columns** |

### New Features Created:
- ⏰ **Temporal:** Hour, Day_Num, Month, Year, Season, Is_Weekend, Is_Night
- 🔢 **Domain:** Crime_Severity_Score (1–10 scale)

### Next Step
Open **`02_ML_Training_Pipeline.ipynb`** to run the clustering algorithms (K-Means, DBSCAN, Hierarchical) and dimensionality reduction (PCA, t-SNE) on this cleaned data! 🤖